# Agentic GitHub Benchmark v2 — issue-driven (`kbench`)

Расширение `benchmark_task_github_skill.ipynb`. Оригинал не изменён.

**Что добавлено:**
1. Расширенный слой GitHub API: issues, comments, trees, compare, pull requests.
2. Новые инструменты агента: `list_files`, `search_code`, `read_file(branch, диапазон строк)`,
   `patch_file`, `delete_file`, `list_issues`, `get_issue`, `create_issue_branch`,
   `comment_issue`, `report_work`, `diff_vs_base`.
3. Новая задача `github_issue_resolution`: агент читает issue, делает изменения
   в отдельной ветке `<slug>/issue-<N>` и комментирует issue ссылкой на ветку.
4. Жёсткий guard: запись разрешена только в ветки с префиксом slug модели.
5. Ретраи и обработка rate limit, обрезка длинных ответов инструментов.

**Требования к токену:** fine-grained PAT c `Contents: RW`, **`Issues: RW`**,
`Pull requests: RW` (опционально), `Metadata: R`.

In [ ]:
import base64
import functools
import json
import re
import time

import requests

GITHUB_OWNER = "mlaa4ml"
GITHUB_REPO = "KaggleModelsRepo"
GITHUB_BASE_BRANCH = "main"
GITHUB_API = "https://api.github.com"

try:
    from kaggle_secrets import UserSecretsClient
    GITHUB_TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception as e:  # noqa: BLE001
    GITHUB_TOKEN = None
    print(f"Нет GITHUB_TOKEN из Kaggle Secrets ({e}). Add-ons -> Secrets.")

assert GITHUB_TOKEN, "Нужен GITHUB_TOKEN"

GITHUB_HEADERS = {
    "Authorization": f"Bearer {GITHUB_TOKEN}",
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28",
}

REPO_PATH = f"/repos/{GITHUB_OWNER}/{GITHUB_REPO}"
REPO_URL = f"https://github.com/{GITHUB_OWNER}/{GITHUB_REPO}"
MAX_TOOL_OUTPUT = 12000  # символов: длинные ответы режем, чтобы не жечь контекст

## 1. Транспорт: ретраи, rate limit, пагинация

In [ ]:
def _gh_request(method, path, retries=3, **kwargs):
    """Один запрос к GitHub API с ретраями на 5xx и вторичный rate limit."""
    last = None
    for attempt in range(retries):
        resp = requests.request(
            method, f"{GITHUB_API}{path}", headers=GITHUB_HEADERS, timeout=30, **kwargs
        )
        last = resp
        if resp.status_code < 500 and resp.status_code != 403:
            return resp
        if resp.status_code == 403 and "rate limit" not in resp.text.lower():
            return resp  # это отказ по правам, ретраить бессмысленно
        time.sleep(2 ** attempt)
    return last


def _gh_paginate(path, params=None, max_pages=10):
    """Собирает все страницы списочного эндпоинта."""
    params = dict(params or {})
    params.setdefault("per_page", 100)
    items, page = [], 1
    while page <= max_pages:
        params["page"] = page
        resp = _gh_request("GET", path, params=params)
        if resp.status_code != 200:
            raise RuntimeError(f"{resp.status_code}: {resp.text[:300]}")
        batch = resp.json()
        if not batch:
            break
        items.extend(batch)
        if len(batch) < params["per_page"]:
            break
        page += 1
    return items


def _clip(text):
    text = str(text)
    if len(text) <= MAX_TOOL_OUTPUT:
        return text
    return text[:MAX_TOOL_OUTPUT] + f"\n...[обрезано, всего {len(text)} символов]"


def _err(label, resp):
    return f"ERROR: {label} failed ({resp.status_code}): {resp.text[:300]}"

## 2. Расширенный слой GitHub: contents, tree, issues, PR

In [ ]:
def gh_list_branches():
    try:
        return ", ".join(b["name"] for b in _gh_paginate(f"{REPO_PATH}/branches"))
    except RuntimeError as e:
        return f"ERROR: list_branches failed: {e}"


def gh_branch_sha(branch):
    resp = _gh_request("GET", f"{REPO_PATH}/git/ref/heads/{branch}")
    return resp.json()["object"]["sha"] if resp.status_code == 200 else None


def gh_create_branch(branch_name, base=None):
    base = base or GITHUB_BASE_BRANCH
    if gh_branch_sha(branch_name):
        return f"Ветка {branch_name} уже существует."
    base_sha = gh_branch_sha(base)
    if not base_sha:
        return f"ERROR: базовая ветка {base} не найдена"
    resp = _gh_request(
        "POST", f"{REPO_PATH}/git/refs",
        json={"ref": f"refs/heads/{branch_name}", "sha": base_sha},
    )
    if resp.status_code not in (200, 201):
        return _err("create_branch", resp)
    return f"Ветка {branch_name} создана от {base} ({base_sha[:7]})."


def gh_list_files(branch, subdir=""):
    """Рекурсивный листинг файлов ветки (git/trees?recursive=1)."""
    sha = gh_branch_sha(branch)
    if not sha:
        return f"ERROR: ветка {branch} не найдена"
    resp = _gh_request("GET", f"{REPO_PATH}/git/trees/{sha}", params={"recursive": "1"})
    if resp.status_code != 200:
        return _err("list_files", resp)
    rows = [
        f"{n['path']} ({n.get('size', 0)} B)"
        for n in resp.json().get("tree", [])
        if n["type"] == "blob" and n["path"].startswith(subdir)
    ]
    return _clip("\n".join(rows) or "(пусто)")


def gh_read_file(path, branch, start_line=1, max_lines=None):
    resp = _gh_request("GET", f"{REPO_PATH}/contents/{path}", params={"ref": branch})
    if resp.status_code != 200:
        return f"ERROR: {path} not found on branch {branch} ({resp.status_code})"
    data = resp.json()
    if data.get("encoding") != "base64":
        return f"ERROR: unexpected encoding {data.get('encoding')!r} for {path}"
    text = base64.b64decode(data["content"]).decode("utf-8", errors="replace")
    if start_line > 1 or max_lines:
        lines = text.splitlines()
        end = start_line - 1 + max_lines if max_lines else len(lines)
        text = "\n".join(lines[start_line - 1:end])
    return _clip(text)


def gh_search_code(query, branch, subdir=""):
    """Простой grep по дереву ветки (без Search API — он лагает на свежих коммитах)."""
    listing = gh_list_files(branch, subdir)
    if listing.startswith("ERROR:"):
        return listing
    hits = []
    for row in listing.splitlines():
        path = row.rsplit(" (", 1)[0]
        content = gh_read_file(path, branch)
        if content.startswith("ERROR:"):
            continue
        for i, line in enumerate(content.splitlines(), start=1):
            if query.lower() in line.lower():
                hits.append(f"{path}:{i}: {line.strip()[:200]}")
                if len(hits) >= 100:
                    return _clip("\n".join(hits))
    return _clip("\n".join(hits) or f"Ничего не найдено по '{query}'")


def gh_write_file(path, content, branch, message=None):
    existing = _gh_request("GET", f"{REPO_PATH}/contents/{path}", params={"ref": branch})
    sha = existing.json().get("sha") if existing.status_code == 200 else None
    payload = {
        "message": message or f"{path}: {'update' if sha else 'create'} ({branch})",
        "content": base64.b64encode(content.encode("utf-8")).decode("ascii"),
        "branch": branch,
    }
    if sha:
        payload["sha"] = sha
    resp = _gh_request("PUT", f"{REPO_PATH}/contents/{path}", json=payload)
    if resp.status_code not in (200, 201):
        return _err("write_file", resp)
    commit = resp.json().get("commit", {}).get("sha", "")[:7]
    return f"Записано {len(content)} символов в {path} (ветка {branch}), commit {commit}"


def gh_delete_file(path, branch, message=None):
    existing = _gh_request("GET", f"{REPO_PATH}/contents/{path}", params={"ref": branch})
    if existing.status_code != 200:
        return f"ERROR: {path} нет в ветке {branch}"
    resp = _gh_request("DELETE", f"{REPO_PATH}/contents/{path}", json={
        "message": message or f"{path}: delete ({branch})",
        "sha": existing.json()["sha"],
        "branch": branch,
    })
    return f"Удалён {path} из {branch}" if resp.status_code == 200 else _err("delete_file", resp)


def gh_compare(base, head):
    resp = _gh_request("GET", f"{REPO_PATH}/compare/{base}...{head}")
    if resp.status_code != 200:
        return None
    d = resp.json()
    return {
        "ahead_by": d.get("ahead_by", 0),
        "behind_by": d.get("behind_by", 0),
        "files": [f["filename"] for f in d.get("files", [])],
        "commits": [c["commit"]["message"].splitlines()[0] for c in d.get("commits", [])],
    }


# ---------------- Issues ----------------

def gh_list_issues(state="all", labels=None):
    """Все issues репозитория. PR отфильтрованы (GitHub отдаёт их тем же эндпоинтом)."""
    params = {"state": state}
    if labels:
        params["labels"] = labels
    try:
        items = _gh_paginate(f"{REPO_PATH}/issues", params=params)
    except RuntimeError as e:
        return f"ERROR: list_issues failed: {e}"
    rows = []
    for it in items:
        if "pull_request" in it:
            continue
        lbl = ",".join(l["name"] for l in it.get("labels", [])) or "-"
        rows.append(
            f"#{it['number']} [{it['state']}] {it['title']} "
            f"(labels: {lbl}; comments: {it.get('comments', 0)})"
        )
    return _clip("\n".join(rows) or "Открытых/закрытых issues нет.")


def gh_get_issue(number):
    resp = _gh_request("GET", f"{REPO_PATH}/issues/{number}")
    if resp.status_code != 200:
        return _err(f"get_issue #{number}", resp)
    it = resp.json()
    out = [
        f"ISSUE #{it['number']}: {it['title']}",
        f"state: {it['state']} | author: {it['user']['login']} | "
        f"labels: {','.join(l['name'] for l in it.get('labels', [])) or '-'}",
        f"url: {it['html_url']}",
        "",
        "--- BODY ---",
        it.get("body") or "(пусто)",
    ]
    try:
        for c in _gh_paginate(f"{REPO_PATH}/issues/{number}/comments"):
            out += ["", f"--- COMMENT by {c['user']['login']} ---", c.get("body") or ""]
    except RuntimeError:
        out.append("\n(комментарии недоступны)")
    return _clip("\n".join(out))


def gh_comment_issue(number, body):
    resp = _gh_request("POST", f"{REPO_PATH}/issues/{number}/comments", json={"body": body})
    if resp.status_code != 201:
        return _err(f"comment_issue #{number}", resp) + "  (нужен scope Issues: RW)"
    return f"Комментарий опубликован: {resp.json()['html_url']}"


def gh_create_pull_request(head, base, title, body):
    resp = _gh_request("POST", f"{REPO_PATH}/pulls", json={
        "head": head, "base": base, "title": title, "body": body,
    })
    if resp.status_code != 201:
        return _err("create_pull_request", resp) + "  (нужен scope Pull requests: RW)"
    return f"PR создан: {resp.json()['html_url']}"

## 3. Бюджет шагов (как в v1, `functools.wraps` обязателен)

In [ ]:
class StepBudget:
    def __init__(self, limit):
        self.limit, self.used = limit, 0

    def consume(self):
        if self.used >= self.limit:
            return False
        self.used += 1
        return True

    @property
    def remaining(self):
        return max(0, self.limit - self.used)


def with_budget(tool_fn, budget):
    """functools.wraps обязателен: иначе kbench не построит схему параметров."""
    @functools.wraps(tool_fn)
    def wrapped(*args, **kwargs):
        if not budget.consume():
            return (
                f"ERROR: бюджет шагов на раунд исчерпан ({budget.limit}). "
                "Заверши раунд текстовым ответом."
            )
        try:
            return tool_fn(*args, **kwargs)
        except Exception as e:  # noqa: BLE001 — агент должен видеть текст, а не падение
            return f"ERROR: {type(e).__name__}: {e}"
    return wrapped


STEPS_PER_ROUND = 12
MAX_ROUNDS = 3

## 4. Расширенный набор инструментов агента

Guard: агент может писать **только** в ветки с префиксом собственного slug.
Имя ветки нигде не приходит от модели напрямую — оно вычисляется из номера issue.

In [ ]:
def slugify_model_id(model_id):
    return re.sub(r"[^a-z0-9]+", "-", model_id.lower()).strip("-")


def make_github_tools(slug, budget, allow_issue_write=True):
    """Возвращает список инструментов, замкнутых на slug модели."""
    state = {"branch": slug}  # текущая рабочая ветка агента

    def _guard(branch):
        if branch != slug and not branch.startswith(slug + "/"):
            raise PermissionError(f"запись в ветку {branch} запрещена (не твой префикс)")
        return branch

    def list_branches() -> str:
        """List all branch names in the repository."""
        return gh_list_branches()

    def create_branch() -> str:
        """Create your main assigned branch (idempotent)."""
        state["branch"] = slug
        return gh_create_branch(slug)

    def create_issue_branch(issue_number: int) -> str:
        """Create and select a dedicated branch for an issue: <your-slug>/issue-<N>."""
        name = _guard(f"{slug}/issue-{int(issue_number)}")
        state["branch"] = name
        return gh_create_branch(name) + f" Текущая рабочая ветка: {name}"

    def list_files(subdir: str = "") -> str:
        """List all files in the base branch, optionally filtered by path prefix."""
        return gh_list_files(GITHUB_BASE_BRANCH, subdir)

    def read_file(path: str, from_my_branch: bool = False,
                  start_line: int = 1, max_lines: int = 0) -> str:
        """Read a file. from_my_branch=True reads your working branch instead of base.
        Use start_line/max_lines to read only a slice of a large file."""
        branch = state["branch"] if from_my_branch else GITHUB_BASE_BRANCH
        return gh_read_file(path, branch, start_line, max_lines or None)

    def search_code(query: str, subdir: str = "") -> str:
        """Grep the base branch for a substring. Returns 'path:line: text' matches."""
        return gh_search_code(query, GITHUB_BASE_BRANCH, subdir)

    def write_file(path: str, content: str, commit_message: str = "") -> str:
        """Create or overwrite a file with FULL content on your working branch."""
        return gh_write_file(path, content, _guard(state["branch"]), commit_message or None)

    def patch_file(path: str, old_text: str, new_text: str) -> str:
        """Replace the first occurrence of old_text with new_text in a file
        on your working branch. Cheaper and safer than rewriting a whole file."""
        branch = _guard(state["branch"])
        current = gh_read_file(path, branch)
        if current.startswith("ERROR:"):
            return current
        if old_text not in current:
            return "ERROR: old_text не найден — прочитай файл и повтори точную подстроку"
        updated = current.replace(old_text, new_text, 1)
        return gh_write_file(path, updated, branch, f"{path}: patch")

    def delete_file(path: str) -> str:
        """Delete a file on your working branch."""
        return gh_delete_file(path, _guard(state["branch"]))

    def diff_vs_base() -> str:
        """Show how your working branch differs from the base branch."""
        cmp = gh_compare(GITHUB_BASE_BRANCH, state["branch"])
        if cmp is None:
            return f"ERROR: не удалось сравнить {state['branch']} с {GITHUB_BASE_BRANCH}"
        return (
            f"branch={state['branch']} ahead_by={cmp['ahead_by']} behind_by={cmp['behind_by']}\n"
            f"files: {', '.join(cmp['files']) or '-'}\n"
            f"commits: {'; '.join(cmp['commits']) or '-'}"
        )

    def list_issues(state_filter: str = "all", labels: str = "") -> str:
        """List ALL issues (open and closed) with number, state, title and labels."""
        return gh_list_issues(state_filter, labels or None)

    def get_issue(issue_number: int) -> str:
        """Read the full body and all comments of one issue."""
        return gh_get_issue(int(issue_number))

    def comment_issue(issue_number: int, body: str) -> str:
        """Post a comment on an issue."""
        return gh_comment_issue(int(issue_number), body)

    def report_work(issue_number: int, summary: str) -> str:
        """Post a structured report on the issue: link to your branch, diff link,
        changed files and your summary. Use this to close the loop on a task."""
        branch = state["branch"]
        cmp = gh_compare(GITHUB_BASE_BRANCH, branch) or {"files": [], "commits": [], "ahead_by": 0}
        body = (
            f"### Работа по issue #{int(issue_number)}\n\n"
            f"{summary}\n\n"
            f"**Ветка:** [`{branch}`]({REPO_URL}/tree/{branch})\n"
            f"**Diff:** {REPO_URL}/compare/{GITHUB_BASE_BRANCH}...{branch}\n"
            f"**Коммитов впереди `{GITHUB_BASE_BRANCH}`:** {cmp['ahead_by']}\n"
            f"**Изменённые файлы:** "
            + (", ".join(f"`{f}`" for f in cmp["files"]) or "—")
            + "\n\n<sub>Автоматический отчёт агента.</sub>"
        )
        return gh_comment_issue(int(issue_number), body)

    def open_pull_request(title: str, body: str = "") -> str:
        """Open a pull request from your working branch into the base branch."""
        return gh_create_pull_request(state["branch"], GITHUB_BASE_BRANCH, title, body)

    tools = [
        list_branches, create_branch, create_issue_branch,
        list_files, read_file, search_code,
        write_file, patch_file, delete_file, diff_vs_base,
    ]
    if allow_issue_write:
        tools += [list_issues, get_issue, comment_issue, report_work, open_pull_request]
    else:
        tools += [list_issues, get_issue]
    return [with_budget(t, budget) for t in tools]

## 5. Задача `github_issue_resolution`

Успех засчитывается **только** если одновременно:
1. ветка `<slug>/issue-<N>` существует и опережает `main` хотя бы на 1 коммит;
2. в issue есть комментарий со ссылкой на эту ветку.

In [ ]:
import kaggle_benchmarks as kbench  # noqa: E402
import pandas as pd  # noqa: E402

ISSUE_SYSTEM_PROMPT = (
    "Ты — инженер-агент, работающий с реальным GitHub-репозиторием только через "
    "инструменты. Твой цикл работы: прочитать issue -> изучить релевантные файлы "
    "(list_files/search_code/read_file) -> создать ветку под issue "
    "(create_issue_branch) -> внести изменения (write_file/patch_file) -> "
    "проверить (diff_vs_base) -> отчитаться в issue (report_work). "
    "Никогда не выдумывай содержимое файлов — сначала читай."
)

ISSUE_ROUNDS_LOG = []


def run_rounds(llm, make_round_tools, task_check_fn, task_context):
    self_report = None
    for round_num in range(1, MAX_ROUNDS + 1):
        budget = StepBudget(STEPS_PER_ROUND)
        tools = make_round_tools(budget)
        if round_num == 1:
            msg = (
                f"{task_context}\n\nУ тебя до {STEPS_PER_ROUND} вызовов инструментов "
                f"на раунд и до {MAX_ROUNDS} раундов."
            )
        else:
            msg = (
                f"Раунд {round_num}/{MAX_ROUNDS}. Новый бюджет: {STEPS_PER_ROUND} "
                "вызовов. Продолжай с того места, где остановился(-ась)."
            )
        if round_num == MAX_ROUNDS:
            msg += (
                "\n\nПОСЛЕДНИЙ РАУНД. В конце добавь блок:\n===SELF_REPORT===\n"
                "готово_процентов: <0-100>\nосталось_сделать: <кратко>\n"
                "нужно_ещё_шагов: <число>\n===END_SELF_REPORT==="
            )
        response = llm.prompt(msg, tools=tools)
        m = re.search(r"===SELF_REPORT===(.*?)===END_SELF_REPORT===", response, re.DOTALL)
        if m:
            self_report = m.group(1).strip()
        if task_check_fn():
            return True, round_num, self_report
    return False, MAX_ROUNDS, self_report


def check_issue_solved(slug, issue_number):
    branch = f"{slug}/issue-{issue_number}"
    cmp = gh_compare(GITHUB_BASE_BRANCH, branch)
    if not cmp or cmp["ahead_by"] < 1:
        return False
    try:
        comments = _gh_paginate(f"{REPO_PATH}/issues/{issue_number}/comments")
    except RuntimeError:
        return False
    return any(branch in (c.get("body") or "") for c in comments)


@kbench.task(name="github_issue_resolution")
def github_issue_resolution_eval(llm, model_id: str, issue_number: int) -> bool:
    slug = slugify_model_id(model_id)
    issue_number = int(issue_number)
    branch = f"{slug}/issue-{issue_number}"

    task_context = (
        f"Репозиторий: {GITHUB_OWNER}/{GITHUB_REPO} (база: {GITHUB_BASE_BRANCH}).\n"
        f"Задача: разбери issue #{issue_number}.\n"
        "1. get_issue — прочитай задачу целиком, включая комментарии.\n"
        "2. list_files / search_code / read_file — изучи затронутый код.\n"
        f"3. create_issue_branch({issue_number}) — создай ветку {branch}.\n"
        "4. write_file / patch_file — внеси изменения ТОЛЬКО в эту ветку.\n"
        "5. diff_vs_base — убедись, что изменения на месте.\n"
        f"6. report_work({issue_number}, '<что сделано>') — оставь комментарий "
        "в issue со ссылкой на ветку."
    )

    with kbench.chats.new(f"{model_id}::github_issue_resolution#{issue_number}") as chat:
        kbench.user.send(ISSUE_SYSTEM_PROMPT)
        solved, rounds_used, self_report = run_rounds(
            llm,
            lambda b: make_github_tools(slug, b),
            lambda: check_issue_solved(slug, issue_number),
            task_context,
        )

    ISSUE_ROUNDS_LOG.append({
        "model_id": model_id, "issue": issue_number, "branch": branch,
        "solved": solved, "rounds_used": rounds_used, "self_report": self_report,
    })
    if not solved:
        kbench.assertions.assert_fail(
            expectation=f"issue #{issue_number}: нет ветки {branch} с коммитами "
                        "и/или комментария со ссылкой на неё"
        )
    return solved

## 6. Прогон: каждая модель × каждый открытый issue

In [ ]:
MODELS = [
    "anthropic/claude-opus-5@default",
    "google/gemini-3.5-flash-lite",
    "openai/gpt-5.4-nano-2026-03-17",
]

open_issues = [
    it["number"]
    for it in _gh_paginate(f"{REPO_PATH}/issues", params={"state": "open"})
    if "pull_request" not in it
]
print("Открытые issues:", open_issues)

completed = []
for model_id in MODELS:
    for num in open_issues:
        results = github_issue_resolution_eval.evaluate(
            llm=[kbench.llms[model_id]],
            evaluation_data=pd.DataFrame([{"model_id": model_id, "issue_number": num}]),
            on_failure="continue",
            max_attempts=1,
        )
        df = results.completed_runs.as_dataframe()
        df["model_id"], df["issue"] = model_id, num
        completed.append(df)
        print(f"{model_id} / issue #{num}: "
              f"{len(results.completed_runs)} ok, {len(results.errored_runs)} err")

issue_rounds_df = pd.DataFrame(ISSUE_ROUNDS_LOG)
issue_rounds_df

## 7. Смоук-тест инструментов без LLM

Дешёвая проверка, что расширенный слой и scope токена в порядке —
до того, как тратить деньги на модели.

In [ ]:
def smoke_test(slug="anthropic-claude-opus-5-default"):
    checks = {
        "branches": gh_list_branches(),
        "files(main)": gh_list_files(GITHUB_BASE_BRANCH)[:300],
        "issues(all)": gh_list_issues("all")[:500],
        "compare": gh_compare(GITHUB_BASE_BRANCH, slug),
    }
    for k, v in checks.items():
        ok = not (isinstance(v, str) and v.startswith("ERROR:"))
        print(f"[{'OK ' if ok else 'FAIL'}] {k}: {str(v)[:250]}\n")
    print("Если issues -> FAIL 403, у токена нет scope 'Issues: Read and write'.")


smoke_test()